# Rehabilitacijski robot ? sposobnost v4

Urejena verzija notebooka za snemanje trajektorij, u?enje CMP signalov in izvedbo rehabilitacijskega zaporedja ?ogica 1 ? 2 ? 3. Projektne datoteke so lo?ene tako, da je koda v `src`, trajektorije v `src/trajectories`, slike v `pictures`, ?lanek pa v `report`.

`SPOSOBNOST = 0` pomeni najve? pomo?i robota. `SPOSOBNOST = 100` pomeni najmanj pomo?i robota.


# Inicializacija robota


In [ ]:
import sys
import time
import pickle
import threading
from pathlib import Path

import rospy
import numpy as np
import matplotlib.pyplot as plt

# Notebook lahko za?enemo iz korena projekta ali neposredno iz mape src.
SRC_DIR = Path.cwd()
if not (SRC_DIR / "utils.py").exists() and (SRC_DIR / "src" / "utils.py").exists():
    SRC_DIR = SRC_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import utils
from utils import SoftSetJointCompliance
from dmp import DMP

from robotblockset.ros.franka import panda_ros
from robotblockset.ros.grippers_ros import PandaGripper

ns = "pingvin_1"
rospy.init_node(ns)
r = panda_ros(ns=ns, control_strategy="JointImpedance", init_node=False)
g = PandaGripper(namespace=ns, robot=r)

r.ErrorRecovery()
r.ResetCurrentTarget()

print("Robot je inicializiran.")


# SKLOP 1 — Nastavitve

V tej celici se običajno spreminjajo samo imena datotek, število ponovitev in `SPOSOBNOST`.

`SPOSOBNOST = 0` pomeni največ pomoči robota. `SPOSOBNOST = 100` pomeni najmanj pomoči robota.


In [ ]:
DATA_DIR = SRC_DIR / "trajectories"
RESULTS_DIR = SRC_DIR / "results"
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

IME_ZACETNE_TOCKE_PRIVZETO = DATA_DIR / "zacetna_tocka_zogice.npz"

DATOTEKA_PREMIK = DATA_DIR / "premik_v1.npz"
CMP_DATOTEKA_PREMIK = DATA_DIR / "cmp_premik_v1.npz"

DATOTEKE_ZOGICA = [
    DATA_DIR / "zogica_1.npz",
    DATA_DIR / "zogica_2.npz",
    DATA_DIR / "zogica_3.npz",
]

CMP_DATOTEKE_ZOGICA = [
    DATA_DIR / "cmp_zogica_1_naprej.npz",
    DATA_DIR / "cmp_zogica_2_naprej.npz",
    DATA_DIR / "cmp_zogica_3_naprej.npz",
]

CMP_DATOTEKE_ZOGICA_NAZAJ = [
    DATA_DIR / "cmp_zogica_1_nazaj.npz",
    DATA_DIR / "cmp_zogica_2_nazaj.npz",
    DATA_DIR / "cmp_zogica_3_nazaj.npz",
]

STEVILO_PONOVITEV_PREMIK = 10
STEVILO_CIKLOV_ZOGICA = 10

NUM_WEIGHTS = 35
CAS_PREMIKA_NA_ZACETEK = 5.0
CAS_POCITKA = 0.5
FREKVENCA_SNEMANJA = 100
ZAMIK_PRED_SNEMANJEM = 1.0

SPOSOBNOST = 50
if not (0 <= SPOSOBNOST <= 100):
    raise ValueError("SPOSOBNOST mora biti med 0 in 100.")

delez_pomoci = (100 - SPOSOBNOST) / 100.0

JOINT_SOFT_ROCNO = 0.0
JOINT_SOFT_MIN = 0.005
JOINT_SOFT_MAX = 0.08
JOINT_SOFT_CMP = JOINT_SOFT_MIN + delez_pomoci * (JOINT_SOFT_MAX - JOINT_SOFT_MIN)

CMP_GAIN_MIN = 0.05
CMP_GAIN_MAX = 0.35
CMP_GAIN = CMP_GAIN_MIN + delez_pomoci * (CMP_GAIN_MAX - CMP_GAIN_MIN)

TAU_LIMIT = 4.0
RAMP_DELEZ_CMP = 0.12
PROSTA_ZADNJA_DVA = False
UPORABI_CMP_PRI_IZVEDBI = True
IZVEDI_NAZAJ_PO_VSAKI_ZOGICI = True

DRZI_ZADNJO_TOCKO_PO_IZVEDBI = True
MIN_JOINT_SOFT_IZVEDBA_VARNO = 0.25
CAS_DRZANJA_ZADNJE_TOCKE = 1.0
CAS_DRZANJA_PRED_GIBOM = 0.6
FREKVENCA_DRZANJA_ZADNJE_TOCKE = 100

# Stara imena ostanejo zaradi kompatibilnosti z ?e obstoje?imi celicami/grafi.
UTRDI_PO_VSAKI_IZVEDBI = DRZI_ZADNJO_TOCKO_PO_IZVEDBI
CAS_UTRDITVE_PO_IZVEDBI = CAS_DRZANJA_ZADNJE_TOCKE

OUT_LOG = RESULTS_DIR / "rezultati_sposobnost_v4.pkl"

if len(DATOTEKE_ZOGICA) != 3:
    raise ValueError("Za rehabilitacijski cikel morajo biti v DATOTEKE_ZOGICA vpisane to?no tri trajektorije.")

if len(CMP_DATOTEKE_ZOGICA) != len(DATOTEKE_ZOGICA):
    raise ValueError("CMP_DATOTEKE_ZOGICA mora imeti enako ?tevilo elementov kot DATOTEKE_ZOGICA.")

if len(CMP_DATOTEKE_ZOGICA_NAZAJ) != len(DATOTEKE_ZOGICA):
    raise ValueError("CMP_DATOTEKE_ZOGICA_NAZAJ mora imeti enako ?tevilo elementov kot DATOTEKE_ZOGICA.")

print("DATA_DIR:", DATA_DIR)
print("DATOTEKA_PREMIK:", DATOTEKA_PREMIK)
print("CMP_DATOTEKA_PREMIK:", CMP_DATOTEKA_PREMIK)
print("DATOTEKE_ZOGICA:", DATOTEKE_ZOGICA)
print("CMP_DATOTEKE_ZOGICA:", CMP_DATOTEKE_ZOGICA)
print("CMP_DATOTEKE_ZOGICA_NAZAJ:", CMP_DATOTEKE_ZOGICA_NAZAJ)
print("SPOSOBNOST:", SPOSOBNOST)
print("JOINT_SOFT_CMP:", round(JOINT_SOFT_CMP, 4))
print("CMP_GAIN:", round(CMP_GAIN, 4))
print("TAU_LIMIT:", TAU_LIMIT)
print("DRZI_ZADNJO_TOCKO_PO_IZVEDBI:", DRZI_ZADNJO_TOCKO_PO_IZVEDBI)
print("CAS_DRZANJA_ZADNJE_TOCKE:", CAS_DRZANJA_ZADNJE_TOCKE)
print("MIN_JOINT_SOFT_IZVEDBA_VARNO:", MIN_JOINT_SOFT_IZVEDBA_VARNO)
print("PROSTA_ZADNJA_DVA:", PROSTA_ZADNJA_DVA)
print("IZVEDI_NAZAJ_PO_VSAKI_ZOGICI:", IZVEDI_NAZAJ_PO_VSAKI_ZOGICI)


## Funkcije za datoteke, snemanje, DMP in CMP


In [ ]:
def ime_npz(ime):
    besedilo = str(ime).strip()
    if not besedilo:
        raise ValueError("Ime ne sme biti prazno.")
    pot = Path(besedilo)
    if pot.suffix.lower() != ".npz":
        pot = pot.with_suffix(".npz")
    if not pot.is_absolute() and pot.parent == Path("."):
        pot = DATA_DIR / pot.name
    return str(pot)


def preberi_navor(robot):
    if hasattr(robot, "state") and hasattr(robot.state, "tau_J"):
        return np.asarray(robot.state.tau_J).copy()
    if hasattr(robot, "state") and hasattr(robot.state, "tau_J_d"):
        return np.asarray(robot.state.tau_J_d).copy()
    if hasattr(robot, "trq"):
        return np.asarray(robot.trq).copy()
    if hasattr(robot, "GetJointTrq"):
        return np.asarray(robot.GetJointTrq()).copy()
    return np.zeros(robot.nj)


def nastavi_togo_vodenje(robot, reset=True, oznaka=""):
    if reset:
        robot.ErrorRecovery()
        robot.ResetCurrentTarget()
    SoftSetJointCompliance(robot, robot._franka_default.JointCompliance.K, 4)
    robot.SetJointSoft(1)
    if reset:
        robot.ResetCurrentTarget()
    if oznaka:
        print(oznaka)


def pripravi_robota_za_rocno_snemanje(robot):
    robot.ErrorRecovery()
    robot.ResetCurrentTarget()
    robot.SetJointCompliant()
    robot.SetJointSoft(JOINT_SOFT_ROCNO)
    robot.ResetCurrentTarget()
    print("Robot je podajen za ročno snemanje.")


def pripravi_robota_za_premik_na_zacetek(robot):
    nastavi_togo_vodenje(robot, reset=True, oznaka="Robot je pripravljen za premik na začetno lego.")


def pripravi_robota_za_ucenje_cmp(robot):
    nastavi_togo_vodenje(robot, reset=True, oznaka="Robot je pripravljen za avtomatsko izvedbo in snemanje CMP.")


def pripravi_robota_za_izvedbo_s_cmp(robot):
    robot.ErrorRecovery()
    robot.ResetCurrentTarget()

    # Ta verzija robota med rehabilitacijo ne preklopi v popolno podajnost.
    # Mehkejši občutek dobimo z zmanjšano togostjo, vendar z varnim minimumom,
    # da robot na začetku in med premori ne pade dol.
    joint_soft_varno = max(float(JOINT_SOFT_CMP), float(MIN_JOINT_SOFT_IZVEDBA_VARNO))

    try:
        K0 = np.asarray(robot._franka_default.JointCompliance.K, dtype=float).copy()
        K = K0 * joint_soft_varno
        if PROSTA_ZADNJA_DVA and robot.nj >= 2:
            K[-2:] = K0[-2:] * joint_soft_varno
        SoftSetJointCompliance(robot, K, 4)
    except Exception as e:
        print("Dodatna nastavitev togosti ni bila uporabljena:", e)

    try:
        robot.SetJointSoft(joint_soft_varno)
    except Exception as e:
        print("SetJointSoft ni uspel:", e)

    robot.ResetCurrentTarget()
    print("Robot je pripravljen za izvedbo s CMP pomočjo brez preklopa v podajno stanje.")


def shrani_trenutno_tocko(robot, ime_tocke):
    ime_datoteke = ime_npz(ime_tocke)
    robot.GetState()
    q = np.asarray(robot.q).copy()
    dq = np.asarray(robot.qdot).copy()
    tau = preberi_navor(robot)
    np.savez(
        ime_datoteke,
        q=q,
        dq=dq,
        tau=tau,
        cas=time.time(),
        ime_tocke=ime_datoteke,
    )
    print("Začetna točka je shranjena:", ime_datoteke)
    print("q =", q)
    return q


def nalozi_tocko(ime_tocke):
    ime_datoteke = ime_npz(ime_tocke)
    pot = Path(ime_datoteke)
    if not pot.exists():
        raise FileNotFoundError(f"Manjka začetna točka: {ime_datoteke}")
    podatki = np.load(ime_datoteke, allow_pickle=True)
    if "q" not in podatki.files:
        raise ValueError(f"Datoteka {ime_datoteke} ne vsebuje spremenljivke q.")
    return np.asarray(podatki["q"]).copy()


def posnemi_trajektorijo_z_zacetno_tocko(robot, ime_trajektorije, ime_tocke, frekvenca=100, zamik=1.0):
    ime_trajektorije = ime_npz(ime_trajektorije)
    ime_tocke = ime_npz(ime_tocke)
    q_zacetek = nalozi_tocko(ime_tocke)

    pripravi_robota_za_premik_na_zacetek(robot)
    print("Premik v začetno točko:", ime_tocke)
    robot.JMove(q_zacetek, CAS_PREMIKA_NA_ZACETEK)
    time.sleep(CAS_POCITKA)

    pripravi_robota_za_rocno_snemanje(robot)

    ukaz = input("Za začetek snemanja vpiši 'n': ").strip().lower()
    while ukaz != "n":
        ukaz = input("Za začetek snemanja vpiši 'n': ").strip().lower()

    print(f"Snemanje se začne čez {zamik:.1f} s.")
    time.sleep(float(zamik))

    stop = {"value": False}

    def cakaj_konec():
        while True:
            ukaz_stop = input("Za konec snemanja vpiši 'k': ").strip().lower()
            if ukaz_stop == "k":
                stop["value"] = True
                break

    nit = threading.Thread(target=cakaj_konec, daemon=True)
    nit.start()

    dt = 1.0 / frekvenca
    tt = []
    q = []
    dq = []
    tau = []

    print("SNEMAM. Vodi robota po trajektoriji. Za konec vpiši 'k'.")
    start = time.monotonic()
    naslednji = start

    while not stop["value"]:
        robot.GetState()
        tt.append(time.monotonic() - start)
        q.append(np.asarray(robot.q).copy())
        dq.append(np.asarray(robot.qdot).copy())
        tau.append(preberi_navor(robot))
        naslednji += dt
        while time.monotonic() < naslednji and not stop["value"]:
            time.sleep(0.0005)

    tt = np.asarray(tt)
    q = np.asarray(q)
    dq = np.asarray(dq)
    tau = np.asarray(tau)

    if len(tt) > 0:
        tt = tt - tt[0]

    np.savez(
        ime_trajektorije,
        tt=tt,
        qt=q,
        dqt=dq,
        tau=tau,
        frekvenca=frekvenca,
        ime_zacetne_tocke=ime_tocke,
        q_zacetek_shranjen=q_zacetek,
    )

    print("Snemanje končano.")
    print("Število vzorcev:", len(tt))
    print("Trajanje:", round(float(tt[-1]), 3) if len(tt) else 0.0, "s")
    print("Shranjeno:", ime_trajektorije)
    return tt, q, dq, tau


In [ ]:
def pocakaj_do(start, t_naslednji):
    cilj = start + float(t_naslednji)
    while True:
        dt = cilj - time.monotonic()
        if dt <= 0:
            break
        time.sleep(min(dt, 0.001))


def q_v_tcp_varno(robot, q_pot):
    q_pot = np.asarray(q_pot)
    N = q_pot.shape[0]
    p_pot = np.zeros((N, 3))
    for i in range(N):
        try:
            x_i = np.asarray(robot.DKin(q_pot[i], out="x", task_space="World")).reshape(-1)
            p_pot[i, :] = x_i[:3]
        except Exception:
            p_pot[i, :] = np.nan
    return p_pot


def preveri_datoteke_trajektorij(datoteke):
    manjkajo = []
    obstajajo = []
    for ime in datoteke:
        ime_datoteke = ime_npz(ime)
        if Path(ime_datoteke).exists():
            obstajajo.append(ime_datoteke)
        else:
            manjkajo.append(ime_datoteke)
    if manjkajo:
        besedilo = "Manjkajo datoteke trajektorij: " + ", ".join(manjkajo)
        besedilo += ". Popravi imena v nastavitvah ali najprej posnemi trajektorije."
        raise FileNotFoundError(besedilo)
    return obstajajo


def nalozi_trajektorijo_kot_dmp(ime_datoteke, num_weights=35):
    ime_datoteke = ime_npz(ime_datoteke)
    pot = Path(ime_datoteke)
    if not pot.exists():
        raise FileNotFoundError(f"Manjka datoteka trajektorije: {ime_datoteke}")
    podatki = np.load(ime_datoteke, allow_pickle=True)
    if "tt" not in podatki.files:
        raise ValueError(f"Datoteka {ime_datoteke} ne vsebuje tt.")
    if "qt" in podatki.files:
        q = np.asarray(podatki["qt"])
    elif "q" in podatki.files:
        q = np.asarray(podatki["q"])
    else:
        raise ValueError(f"Datoteka {ime_datoteke} ne vsebuje qt ali q.")
    t = np.asarray(podatki["tt"]).reshape(-1)
    if len(t) < 2:
        raise ValueError(f"Trajektorija {ime_datoteke} ima premalo vzorcev.")
    t = t - t[0]
    if "dqt" in podatki.files:
        dq = np.asarray(podatki["dqt"])
    elif "dq" in podatki.files:
        dq = np.asarray(podatki["dq"])
    else:
        dq = np.gradient(q, t, axis=0)
    model = DMP(q.copy(), t.copy(), vel_data=dq.copy(), num_weights=num_weights)
    q_ref, t_ref = model.decode()
    t_ref = np.asarray(t_ref).reshape(-1)
    dq_ref = np.gradient(q_ref, t_ref, axis=0)
    oznaka = Path(ime_datoteke).stem
    return {
        "id": oznaka,
        "oznaka": oznaka,
        "ime_datoteke": ime_datoteke,
        "t_posneto": t,
        "q_posneto": q,
        "dq_posneto": dq,
        "model": model,
        "t_ref": t_ref,
        "q_ref": q_ref,
        "dq_ref": dq_ref,
        "q_zacetek": q_ref[0].copy(),
        "q_konec": q_ref[-1].copy(),
        "smer": "naprej",
    }


def ustvari_povratno_ref(ref):
    q_ref = np.asarray(ref["q_ref"])[::-1].copy()
    t_ref = np.asarray(ref["t_ref"]).reshape(-1).copy()
    t_ref = t_ref - t_ref[0]
    dq_ref = np.gradient(q_ref, t_ref, axis=0)
    ref_nazaj = dict(ref)
    ref_nazaj["id"] = ref["id"] + "_nazaj"
    ref_nazaj["oznaka"] = ref["oznaka"] + "_nazaj"
    ref_nazaj["q_ref"] = q_ref
    ref_nazaj["dq_ref"] = dq_ref
    ref_nazaj["t_ref"] = t_ref
    ref_nazaj["q_zacetek"] = q_ref[0].copy()
    ref_nazaj["q_konec"] = q_ref[-1].copy()
    ref_nazaj["smer"] = "nazaj"
    ref_nazaj.pop("tau_cmp", None)
    ref_nazaj.pop("t_cmp", None)
    ref_nazaj.pop("cmp_datoteka", None)
    ref_nazaj.pop("ucni_poskus", None)
    return ref_nazaj


def interpoliraj_vrstice(y, t_stari, t_novi):
    y = np.asarray(y)
    t_stari = np.asarray(t_stari).reshape(-1)
    t_novi = np.asarray(t_novi).reshape(-1)
    if y.ndim == 1:
        y = y.reshape(-1, 1)
    y_novi = np.zeros((len(t_novi), y.shape[1]))
    for j in range(y.shape[1]):
        y_novi[:, j] = np.interp(t_novi, t_stari, y[:, j])
    return y_novi


def naredi_ramp(N, delez=0.12):
    ramp = np.ones(N)
    n = int(max(1, round(N * float(delez))))
    if n > 1:
        gor = np.linspace(0.0, 1.0, n)
        dol = np.linspace(1.0, 0.0, n)
        ramp[:n] = gor
        ramp[-n:] = np.minimum(ramp[-n:], dol)
    return ramp.reshape(-1, 1)


def pripravi_tau_cmp_za_ref(ref):
    if not UPORABI_CMP_PRI_IZVEDBI:
        return None
    if "tau_cmp" not in ref or "t_cmp" not in ref:
        return None
    tau = interpoliraj_vrstice(ref["tau_cmp"], ref["t_cmp"], ref["t_ref"])
    tau = CMP_GAIN * tau
    tau = tau * naredi_ramp(tau.shape[0], RAMP_DELEZ_CMP)
    if PROSTA_ZADNJA_DVA and tau.shape[1] >= 2:
        tau[:, -2:] = 0.0
    tau = np.clip(tau, -TAU_LIMIT, TAU_LIMIT)
    return tau


def ime_cmp_za_zogico(ref, indeks, nazaj=False):
    tabela = CMP_DATOTEKE_ZOGICA_NAZAJ if nazaj else CMP_DATOTEKE_ZOGICA
    koncnica = "_nazaj" if nazaj else ""
    if isinstance(tabela, dict):
        return ime_npz(tabela.get(ref["ime_datoteke"], f"cmp_zogica{indeks}{koncnica}.npz"))
    if isinstance(tabela, (list, tuple)):
        if indeks - 1 >= len(tabela):
            raise IndexError("Seznam CMP datotek nima dovolj imen za vse trajektorije žogice.")
        return ime_npz(tabela[indeks - 1])
    raise TypeError("CMP datoteke za žogico morajo biti seznam ali slovar.")


def drzi_tocko_stabilno(robot, q_cilj, cas=None, oznaka=""):
    if cas is None:
        cas = CAS_DRZANJA_ZADNJE_TOCKE

    q_cilj = np.asarray(q_cilj).copy()
    dq_nic = np.zeros(robot.nj)
    tau_nic = np.zeros(robot.nj)

    try:
        nastavi_togo_vodenje(robot, reset=True)
        start = time.monotonic()
        dt = 1.0 / float(FREKVENCA_DRZANJA_ZADNJE_TOCKE)
        naslednji = start

        while time.monotonic() - start < float(cas):
            robot.GoTo_q(q_cilj, dq_nic, tau_nic, 0)
            naslednji += dt
            pocakaj_do(start, naslednji - start)

        robot.GoTo_q(q_cilj, dq_nic, tau_nic, 0)
        if oznaka:
            print("Robot stabilno drži točko:", oznaka)
    except Exception as e:
        print("Opozorilo: stabilno držanje točke ni uspelo:", e)


def drzi_zadnjo_tocko_trajektorije(robot, ref, tau_zadnji=None, oznaka=""):
    if not DRZI_ZADNJO_TOCKO_PO_IZVEDBI:
        return
    drzi_tocko_stabilno(robot, np.asarray(ref["q_ref"])[-1], cas=CAS_DRZANJA_ZADNJE_TOCKE, oznaka=oznaka)


def utrdi_robota_v_trenutni_legi(robot):
    try:
        robot.GetState()
        drzi_tocko_stabilno(robot, np.asarray(robot.q).copy(), cas=CAS_DRZANJA_ZADNJE_TOCKE, oznaka="trenutna lega")
    except Exception as e:
        print("Opozorilo: držanje trenutne lege ni uspelo:", e)


def izvedi_in_posnemi(robot, ref, oznaka, tau_dodatni=None, mehko=False):
    q_ref = np.asarray(ref["q_ref"])
    dq_ref = np.asarray(ref["dq_ref"])
    t_ref = np.asarray(ref["t_ref"]).reshape(-1)
    N = q_ref.shape[0]
    nj = robot.nj

    if tau_dodatni is None:
        tau_dodatni = np.zeros((N, nj))
    else:
        tau_dodatni = np.asarray(tau_dodatni).copy()
        if tau_dodatni.shape[0] != N:
            tau_dodatni = interpoliraj_vrstice(
                tau_dodatni,
                np.linspace(0.0, float(t_ref[-1]), tau_dodatni.shape[0]),
                t_ref,
            )
    if PROSTA_ZADNJA_DVA and tau_dodatni.shape[1] >= 2:
        tau_dodatni[:, -2:] = 0.0

    tt = np.zeros((N, 1))
    q = np.zeros((N, nj))
    dq = np.zeros((N, nj))
    tau = np.zeros((N, nj))
    err_q = np.zeros(N)

    print("Izvajam:", oznaka)
    start = time.monotonic()

    for i in range(N):
        robot.GoTo_q(q_ref[i], dq_ref[i], tau_dodatni[i], 0)
        robot.GetState()
        tt[i, 0] = time.monotonic() - start
        q[i, :] = robot.q
        dq[i, :] = robot.qdot
        tau[i, :] = preberi_navor(robot)
        err_q[i] = np.linalg.norm(q_ref[i] - q[i, :])
        if i < N - 1:
            pocakaj_do(start, t_ref[i + 1])

    p_ref = q_v_tcp_varno(robot, q_ref)
    p = q_v_tcp_varno(robot, q)
    err_p = np.linalg.norm(p_ref - p, axis=1)

    return {
        "oznaka": oznaka,
        "ime_datoteke": ref["ime_datoteke"],
        "smer": ref.get("smer", "naprej"),
        "mehko": bool(mehko),
        "SPOSOBNOST": SPOSOBNOST,
        "JOINT_SOFT_CMP": JOINT_SOFT_CMP,
        "CMP_GAIN": CMP_GAIN,
        "TAU_LIMIT": TAU_LIMIT,
        "t_ref": t_ref.copy(),
        "q_ref": q_ref.copy(),
        "dq_ref": dq_ref.copy(),
        "tt": tt,
        "q": q,
        "dq": dq,
        "tau": tau,
        "tau_dodatni": tau_dodatni.copy(),
        "err_q": err_q,
        "p_ref": p_ref,
        "p": p,
        "err_p": err_p,
    }


def nauci_tau_cmp_iz_meritve(poskus, num_weights=35):
    tau_merjeno = np.asarray(poskus["tau"])
    t_tau = np.asarray(poskus["tt"]).reshape(-1)
    t_tau = t_tau - t_tau[0]
    model_navora = DMP(tau_merjeno.copy(), t_tau.copy(), num_weights=num_weights)
    tau_cmp, t_cmp = model_navora.decode()
    return np.asarray(tau_cmp), np.asarray(t_cmp).reshape(-1)


def shrani_cmp_datoteko(ref, poskus_ucenje, tau_cmp, t_cmp, ime_cmp):
    ime_cmp = ime_npz(ime_cmp)
    np.savez(
        ime_cmp,
        tau_cmp=np.asarray(tau_cmp),
        t_cmp=np.asarray(t_cmp),
        tau_merjeno=np.asarray(poskus_ucenje["tau"]),
        tt_merjeno=np.asarray(poskus_ucenje["tt"]).reshape(-1),
        q_ref=np.asarray(ref["q_ref"]),
        dq_ref=np.asarray(ref["dq_ref"]),
        t_ref=np.asarray(ref["t_ref"]),
        ime_trajektorije=ref["ime_datoteke"],
        oznaka=ref["oznaka"],
        smer=ref.get("smer", "naprej"),
        num_weights=NUM_WEIGHTS,
    )
    print("CMP shranjen:", ime_cmp)
    return ime_cmp


def nalozi_cmp_v_ref(ref, ime_cmp):
    ime_cmp = ime_npz(ime_cmp)
    pot = Path(ime_cmp)
    if not pot.exists():
        raise FileNotFoundError(f"Manjka CMP datoteka: {ime_cmp}")
    podatki = np.load(ime_cmp, allow_pickle=True)
    ref["tau_cmp"] = np.asarray(podatki["tau_cmp"])
    ref["t_cmp"] = np.asarray(podatki["t_cmp"]).reshape(-1)
    ref["cmp_datoteka"] = ime_cmp
    print("Naložen CMP:", ime_cmp, "za", ref["oznaka"])
    return ref


def posnemi_cmp_za_ref(robot, ref, ime_cmp):
    print("UČENJE CMP:", ref["oznaka"])
    pripravi_robota_za_ucenje_cmp(robot)
    print("Premik na začetno lego reference.")
    robot.JMove(ref["q_ref"][0], CAS_PREMIKA_NA_ZACETEK)
    time.sleep(CAS_POCITKA)
    drzi_tocko_stabilno(robot, ref["q_ref"][0], cas=CAS_DRZANJA_PRED_GIBOM, oznaka="začetek CMP " + ref["oznaka"])

    poskus_ucenje = izvedi_in_posnemi(
        robot,
        ref,
        oznaka=f"ucenje_cmp_{ref['oznaka']}",
        tau_dodatni=None,
        mehko=False,
    )

    tau_cmp, t_cmp = nauci_tau_cmp_iz_meritve(poskus_ucenje, num_weights=NUM_WEIGHTS)
    ime_cmp = shrani_cmp_datoteko(ref, poskus_ucenje, tau_cmp, t_cmp, ime_cmp)

    ref["tau_cmp"] = tau_cmp
    ref["t_cmp"] = t_cmp
    ref["cmp_datoteka"] = ime_cmp
    ref["ucni_poskus"] = poskus_ucenje
    drzi_zadnjo_tocko_trajektorije(robot, ref, oznaka="konec CMP " + ref["oznaka"])
    return poskus_ucenje


def premakni_in_drzi_zacetek(robot, ref, oznaka=""):
    q_start = np.asarray(ref["q_ref"])[0]
    pripravi_robota_za_premik_na_zacetek(robot)
    print("Premik na začetno lego reference.")
    robot.JMove(q_start, CAS_PREMIKA_NA_ZACETEK)
    time.sleep(CAS_POCITKA)
    drzi_tocko_stabilno(robot, q_start, cas=CAS_DRZANJA_PRED_GIBOM, oznaka="začetek " + oznaka)


def izvedi_ref_s_cmp(robot, ref, oznaka, premakni_na_zacetek=True):
    if premakni_na_zacetek:
        premakni_in_drzi_zacetek(robot, ref, oznaka=oznaka)
    else:
        drzi_tocko_stabilno(robot, np.asarray(ref["q_ref"])[0], cas=CAS_DRZANJA_PRED_GIBOM, oznaka="prevzem " + oznaka)

    pripravi_robota_za_izvedbo_s_cmp(robot)
    tau_dodatni = pripravi_tau_cmp_za_ref(ref)

    poskus = None
    try:
        poskus = izvedi_in_posnemi(
            robot,
            ref,
            oznaka=oznaka,
            tau_dodatni=tau_dodatni,
            mehko=True,
        )
    finally:
        drzi_zadnjo_tocko_trajektorije(robot, ref, oznaka=oznaka)

    return poskus


# SKLOP 2 — Snemanje in izvedba

## 2.1 Shrani začetno točko

Robota ročno postavi v začetno lego in shrani točko. To točko lahko potem uporabiš pri snemanju `premik` in `zogica` trajektorij.


In [ ]:
pripravi_robota_za_rocno_snemanje(r)
ime_zacetne_tocke = input("Ime začetne točke: ").strip()
if ime_zacetne_tocke == "":
    ime_zacetne_tocke = IME_ZACETNE_TOCKE_PRIVZETO
q_zacetna = shrani_trenutno_tocko(r, ime_zacetne_tocke)


## 2.2 Snemanje trajektorije — premik

Ta del posname osnovno trajektorijo `premik` in jo shrani v `DATOTEKA_PREMIK`.


In [ ]:
zagon = input("Za snemanje PREMIK vpiši 'da': ").strip().lower()
if zagon == "da":
    ime_zacetne_tocke = input("Ime začetne točke: ").strip()
    if ime_zacetne_tocke == "":
        ime_zacetne_tocke = IME_ZACETNE_TOCKE_PRIVZETO
    tt_premik, q_premik, dq_premik, tau_premik = posnemi_trajektorijo_z_zacetno_tocko(
        r,
        DATOTEKA_PREMIK,
        ime_zacetne_tocke,
        frekvenca=FREKVENCA_SNEMANJA,
        zamik=ZAMIK_PRED_SNEMANJEM,
    )
else:
    print("Snemanje PREMIK ni bilo zagnano.")


## 2.3 Snemanje trajektorije — žogica

Ta del posname eno ali več `zogica` trajektorij. Imena se shranijo v `DATOTEKE_ZOGICA` iz nastavitev.


In [ ]:
zagon = input("Za snemanje ŽOGICA trajektorij vpiši 'da': ").strip().lower()
if zagon == "da":
    ime_zacetne_tocke = input("Ime začetne točke: ").strip()
    if ime_zacetne_tocke == "":
        ime_zacetne_tocke = IME_ZACETNE_TOCKE_PRIVZETO

    for ime_trajektorije in DATOTEKE_ZOGICA:
        print("\nSnemanje:", ime_trajektorije)
        posnemi_trajektorijo_z_zacetno_tocko(
            r,
            ime_trajektorije,
            ime_zacetne_tocke,
            frekvenca=FREKVENCA_SNEMANJA,
            zamik=ZAMIK_PRED_SNEMANJEM,
        )
else:
    print("Snemanje ŽOGICA trajektorij ni bilo zagnano.")


## 2.4 Naloži DMP reference iz posnetih trajektorij


In [ ]:
datoteka_premik = preveri_datoteke_trajektorij([DATOTEKA_PREMIK])[0]
ref_premik = nalozi_trajektorijo_kot_dmp(datoteka_premik, num_weights=NUM_WEIGHTS)
ref_premik["tip"] = "premik"
ref_premik["zaporedna"] = 0
ref_premik["cmp_datoteka_predvidena"] = ime_npz(CMP_DATOTEKA_PREMIK)

print("PREMIK:", ref_premik["ime_datoteke"], "q_ref:", ref_premik["q_ref"].shape, "trajanje:", round(float(ref_premik["t_ref"][-1]), 3), "s")

datoteke_zogica = preveri_datoteke_trajektorij(DATOTEKE_ZOGICA)
ref_zogica = []
ref_zogica_nazaj = []
for i, ime in enumerate(datoteke_zogica, start=1):
    ref = nalozi_trajektorijo_kot_dmp(ime, num_weights=NUM_WEIGHTS)
    ref["tip"] = "zogica"
    ref["zaporedna"] = i
    ref["smer"] = "naprej"
    ref["cmp_datoteka_predvidena"] = ime_cmp_za_zogico(ref, i, nazaj=False)
    ref_zogica.append(ref)

    ref_back = ustvari_povratno_ref(ref)
    ref_back["tip"] = "zogica"
    ref_back["zaporedna"] = i
    ref_back["cmp_datoteka_predvidena"] = ime_cmp_za_zogico(ref, i, nazaj=True)
    ref_zogica_nazaj.append(ref_back)

for i, ref in enumerate(ref_zogica, start=1):
    print(
        f"ZOGICA {i} NAPREJ:",
        ref["ime_datoteke"],
        "CMP:", ref["cmp_datoteka_predvidena"],
        "q_ref:", ref["q_ref"].shape,
        "trajanje:", round(float(ref["t_ref"][-1]), 3), "s",
    )
    print(
        f"ZOGICA {i} NAZAJ:",
        ref_zogica_nazaj[i - 1]["ime_datoteke"],
        "CMP:", ref_zogica_nazaj[i - 1]["cmp_datoteka_predvidena"],
        "q_ref:", ref_zogica_nazaj[i - 1]["q_ref"].shape,
        "trajanje:", round(float(ref_zogica_nazaj[i - 1]["t_ref"][-1]), 3), "s",
    )


## 2.5 Posnemi CMP — premik

CMP se posname enkrat za referenco `premik` in shrani v `CMP_DATOTEKA_PREMIK`.


In [ ]:
zagon = input("Za snemanje CMP za PREMIK vpiši 'da': ").strip().lower()
if zagon == "da":
    ucni_poskus_premik = posnemi_cmp_za_ref(r, ref_premik, CMP_DATOTEKA_PREMIK)
elif Path(ime_npz(CMP_DATOTEKA_PREMIK)).exists():
    nalozi_cmp_v_ref(ref_premik, CMP_DATOTEKA_PREMIK)
else:
    print("CMP za PREMIK ni bil posnet in obstoječa CMP datoteka ni bila najdena.")


## 2.6 Posnemi CMP — žogica naprej in nazaj

Za vsako `zogica` trajektorijo se posname CMP za gib naprej in CMP za povratni gib nazaj po isti krivulji. Vsak CMP se shrani v svojo `.npz` datoteko.


In [ ]:
zagon = input("Za snemanje CMP za VSE TRI ŽOGICA trajektorije NAPREJ IN NAZAJ vpiši 'da': ").strip().lower()
ucni_poskusi_zogica = {}
ucni_poskusi_zogica_nazaj = {}

for idx, (ref, ref_nazaj) in enumerate(zip(ref_zogica, ref_zogica_nazaj), start=1):
    ime_cmp = ref.get("cmp_datoteka_predvidena", ime_cmp_za_zogico(ref, idx, nazaj=False))
    ime_cmp_nazaj = ref_nazaj.get("cmp_datoteka_predvidena", ime_cmp_za_zogico(ref, idx, nazaj=True))

    print("\nCMP ŽOGICA", idx, "NAPREJ — trajektorija:", ref["ime_datoteke"], "— cmp:", ime_cmp)
    if zagon == "da":
        ucni_poskusi_zogica[f"zogica{idx}"] = posnemi_cmp_za_ref(r, ref, ime_cmp)
    elif Path(ime_npz(ime_cmp)).exists():
        nalozi_cmp_v_ref(ref, ime_cmp)
    else:
        print("CMP naprej ni posnet in ni najden:", ime_cmp)

    print("\nCMP ŽOGICA", idx, "NAZAJ — trajektorija:", ref_nazaj["ime_datoteke"], "— cmp:", ime_cmp_nazaj)
    if zagon == "da":
        ucni_poskusi_zogica_nazaj[f"zogica{idx}_nazaj"] = posnemi_cmp_za_ref(r, ref_nazaj, ime_cmp_nazaj)
    elif Path(ime_npz(ime_cmp_nazaj)).exists():
        nalozi_cmp_v_ref(ref_nazaj, ime_cmp_nazaj)
    else:
        print("CMP nazaj ni posnet in ni najden:", ime_cmp_nazaj)

print("\nCMP priprava za žogice naprej in nazaj je končana.")


## 2.7 Izvedba — premik 10× z mehko CMP pomočjo


In [ ]:
zagon = input("Za izvedbo PREMIK z mehko CMP pomočjo vpiši 'da': ").strip().lower()
rezultati_premik = []

if zagon == "da":
    for i in range(STEVILO_PONOVITEV_PREMIK):
        oznaka = f"premik_cmp_{i + 1:02d}"
        print("\n---", oznaka, "---")
        poskus = izvedi_ref_s_cmp(r, ref_premik, oznaka)
        poskus["ponovitev"] = i + 1
        rezultati_premik.append(poskus)
else:
    print("Izvedba PREMIK ni bila zagnana.")


## 2.8 Izvedba — žogica 1 → 2 → 3, deset ciklov, s stabilnim vračanjem

V vsakem ciklu se izvedejo trajektorije 1, 2 in 3. Po vsaki trajektoriji se robot, če je `IZVEDI_NAZAJ_PO_VSAKI_ZOGICI = True`, vrne nazaj po isti krivulji z ločenim CMP signalom. Robot pred začetkom, vmes in na koncu vedno stabilno drži ciljno lego in ne preklopi v podajno stanje.


In [ ]:
zagon = input("Za izvedbo ŽOGICA 1→2→3 ciklov z mehko CMP pomočjo vpiši 'da': ").strip().lower()
rezultati_zogica = []

if zagon == "da":
    for cikel in range(STEVILO_CIKLOV_ZOGICA):
        print("\n==============================")
        print(f"CIKEL ŽOGICA {cikel + 1}/{STEVILO_CIKLOV_ZOGICA}: zaporedje 1 → 2 → 3")
        print("==============================")

        for idx, ref in enumerate(ref_zogica, start=1):
            oznaka = f"zogica{idx}_naprej_cikel_{cikel + 1:02d}"
            print("\n---", oznaka, "---")
            poskus = izvedi_ref_s_cmp(r, ref, oznaka, premakni_na_zacetek=True)
            poskus["cikel"] = cikel + 1
            poskus["trajektorija_id"] = idx
            poskus["smer"] = "naprej"
            poskus["cmp_datoteka"] = ref.get("cmp_datoteka", ref.get("cmp_datoteka_predvidena", ""))
            rezultati_zogica.append(poskus)

            if IZVEDI_NAZAJ_PO_VSAKI_ZOGICI:
                ref_nazaj = ref_zogica_nazaj[idx - 1]
                oznaka_nazaj = f"zogica{idx}_nazaj_cikel_{cikel + 1:02d}"
                print("\n---", oznaka_nazaj, "---")
                poskus_nazaj = izvedi_ref_s_cmp(r, ref_nazaj, oznaka_nazaj, premakni_na_zacetek=False)
                poskus_nazaj["cikel"] = cikel + 1
                poskus_nazaj["trajektorija_id"] = idx
                poskus_nazaj["smer"] = "nazaj"
                poskus_nazaj["cmp_datoteka"] = ref_nazaj.get("cmp_datoteka", ref_nazaj.get("cmp_datoteka_predvidena", ""))
                rezultati_zogica.append(poskus_nazaj)

    utrdi_robota_v_trenutni_legi(r)
else:
    print("Izvedba ŽOGICA ni bila zagnana.")


# SKLOP 3 — Grafi in shranjevanje rezultatov

Ta sklop ne premika robota.


In [ ]:
def shrani_rezultate(ime_datoteke=OUT_LOG):
    paket = {
        "verzija": "main_zogica_cmp_sposobnost",
        "datoteka_premik": DATOTEKA_PREMIK,
        "cmp_datoteka_premik": CMP_DATOTEKA_PREMIK,
        "datoteke_zogica": DATOTEKE_ZOGICA,
        "cmp_datoteke_zogica": CMP_DATOTEKE_ZOGICA,
        "cmp_datoteke_zogica_nazaj": CMP_DATOTEKE_ZOGICA_NAZAJ,
        "izvedi_nazaj_po_vsaki_zogici": IZVEDI_NAZAJ_PO_VSAKI_ZOGICI,
        "stevilo_ponovitev_premik": STEVILO_PONOVITEV_PREMIK,
        "stevilo_ciklov_zogica": STEVILO_CIKLOV_ZOGICA,
        "SPOSOBNOST": SPOSOBNOST,
        "JOINT_SOFT_CMP": JOINT_SOFT_CMP,
        "CMP_GAIN": CMP_GAIN,
        "TAU_LIMIT": TAU_LIMIT,
        "UPORABI_CMP_PRI_IZVEDBI": UPORABI_CMP_PRI_IZVEDBI,
        "UTRDI_PO_VSAKI_IZVEDBI": UTRDI_PO_VSAKI_IZVEDBI,
    }
    if "ref_premik" in globals():
        paket["ref_premik"] = ref_premik
    if "ref_zogica" in globals():
        paket["ref_zogica"] = ref_zogica
    if "ref_zogica_nazaj" in globals():
        paket["ref_zogica_nazaj"] = ref_zogica_nazaj
    if "ucni_poskus_premik" in globals():
        paket["ucni_poskus_premik"] = ucni_poskus_premik
    if "ucni_poskusi_zogica" in globals():
        paket["ucni_poskusi_zogica"] = ucni_poskusi_zogica
    if "ucni_poskusi_zogica_nazaj" in globals():
        paket["ucni_poskusi_zogica_nazaj"] = ucni_poskusi_zogica_nazaj
    if "rezultati_premik" in globals():
        paket["rezultati_premik"] = rezultati_premik
    if "rezultati_zogica" in globals():
        paket["rezultati_zogica"] = rezultati_zogica
    with open(ime_datoteke, "wb") as f:
        pickle.dump(paket, f)
    print("Shranjeno:", ime_datoteke)


def izrisi_tcp_poti(rezultati, naslov="TCP poti"):
    if not rezultati:
        print("Ni rezultatov za izris.")
        return
    plt.figure(figsize=(7, 6))
    for d in rezultati:
        p = np.asarray(d.get("p", []))
        if p.size == 0 or np.isnan(p).all():
            continue
        plt.plot(p[:, 0], p[:, 1], alpha=0.75, label=d.get("oznaka", ""))
    plt.xlabel("x [m]")
    plt.ylabel("y [m]")
    plt.title(naslov)
    plt.axis("equal")
    plt.grid(True)
    if len(rezultati) <= 12:
        plt.legend()
    plt.show()


def izrisi_napake(rezultati, naslov="Napaka sledenja"):
    if not rezultati:
        print("Ni rezultatov za izris.")
        return
    oznake = [d.get("oznaka", str(i)) for i, d in enumerate(rezultati)]
    max_err = [1000 * float(np.nanmax(d.get("err_p", np.array([np.nan])))) for d in rezultati]
    mean_err = [1000 * float(np.nanmean(d.get("err_p", np.array([np.nan])))) for d in rezultati]
    x = np.arange(len(rezultati))
    plt.figure(figsize=(10, 4))
    plt.plot(x, max_err, marker="o", label="max TCP napaka")
    plt.plot(x, mean_err, marker="o", label="povprečna TCP napaka")
    plt.xlabel("izvedba")
    plt.ylabel("napaka [mm]")
    plt.title(naslov)
    plt.grid(True)
    plt.legend()
    if len(oznake) <= 20:
        plt.xticks(x, oznake, rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


def izrisi_cmp_ref(ref, naslov=None):
    if "tau_cmp" not in ref:
        print("Referenca nima naloženega CMP signala:", ref.get("oznaka"))
        return
    tau = np.asarray(ref["tau_cmp"])
    t = np.asarray(ref["t_cmp"]).reshape(-1)
    plt.figure(figsize=(10, 4))
    for j in range(tau.shape[1]):
        plt.plot(t, tau[:, j], label=f"sklep {j + 1}")
    plt.xlabel("t [s]")
    plt.ylabel("tau_cmp [Nm]")
    plt.title(naslov or f"CMP signal: {ref.get('oznaka', '')}")
    plt.grid(True)
    plt.legend(ncol=4)
    plt.tight_layout()
    plt.show()


def izrisi_dodan_cmp_izvedbe(rezultati, indeks=0):
    if not rezultati:
        print("Ni rezultatov za izris.")
        return
    d = rezultati[indeks]
    tau = np.asarray(d.get("tau_dodatni", []))
    t = np.asarray(d.get("t_ref", np.arange(len(tau)))).reshape(-1)
    if tau.size == 0:
        print("Izvedba nima shranjenega tau_dodatni.")
        return
    plt.figure(figsize=(10, 4))
    for j in range(tau.shape[1]):
        plt.plot(t, tau[:, j], label=f"sklep {j + 1}")
    plt.xlabel("t [s]")
    plt.ylabel("dodani CMP [Nm]")
    plt.title("Dodani CMP pri izvedbi: " + d.get("oznaka", ""))
    plt.grid(True)
    plt.legend(ncol=4)
    plt.tight_layout()
    plt.show()


In [ ]:
zagon = input("Za shranjevanje vseh rezultatov v pickle vpiši 'da': ").strip().lower()
if zagon == "da":
    shrani_rezultate()
else:
    print("Rezultati niso bili shranjeni v pickle.")


In [ ]:
if "rezultati_premik" in globals() and len(rezultati_premik) > 0:
    izrisi_tcp_poti(rezultati_premik, "PREMIK — TCP poti")
    izrisi_napake(rezultati_premik, "PREMIK — napake")
    izrisi_dodan_cmp_izvedbe(rezultati_premik, indeks=0)

if "rezultati_zogica" in globals() and len(rezultati_zogica) > 0:
    izrisi_tcp_poti(rezultati_zogica, "ŽOGICA — TCP poti")
    izrisi_napake(rezultati_zogica, "ŽOGICA — napake")
    izrisi_dodan_cmp_izvedbe(rezultati_zogica, indeks=0)

if "ref_premik" in globals():
    izrisi_cmp_ref(ref_premik, "CMP — premik")

if "ref_zogica" in globals():
    for ref in ref_zogica:
        izrisi_cmp_ref(ref, "CMP — " + ref.get("oznaka", "zogica"))

if "ref_zogica_nazaj" in globals():
    for ref in ref_zogica_nazaj:
        izrisi_cmp_ref(ref, "CMP — " + ref.get("oznaka", "zogica_nazaj"))
